# 8.4장 텍스트 분류 실습 - 20 뉴스그룹 분류

텍스트 > 피처 벡터화 > 희소 행렬 > ML - 잘 적용되는 알고리즘 : 로지스틱 회귀 ✅, VM, 나이브 베이즈 등

텍스트 기반 분류 > 텍스트 정규화 후 분류 학습/예측/평가

- CountVectorizer , TFIDFVectorizer > 피처벡터화
- GridSearchCV 기반 피처벡터화 하이퍼파라미터 튜닝
- pipeline > 피처벡터화 + 하이퍼파라미터튜닝(GridSearchCV)

## 텍스트 정규화

In [ ]:
from sklearn.datasets import fetch_20newsgroups

news_data = fetch_20newsgroups(subset='all',random_state=156)

In [ ]:
print(news_data.keys())

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])


In [ ]:
import pandas as pd

print('target 클래스 값 & 분포도 \n',pd.Series(news_data.target).value_counts().sort_index())
print('target 클래스의 이름들 \n',news_data.target_names)

target 클래스 값 & 분포도 
 0     799
1     973
2     985
3     982
4     963
5     988
6     975
7     990
8     996
9     994
10    999
11    991
12    984
13    990
14    987
15    997
16    910
17    940
18    775
19    628
Name: count, dtype: int64
target 클래스의 이름들 
 ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


⬆ target 클래스 : 0~19, 20개

In [ ]:
print(news_data.data[0])

From: egreen@east.sun.com (Ed Green - Pixel Cruncher)
Subject: Re: Observation re: helmets
Organization: Sun Microsystems, RTP, NC
Lines: 21
Distribution: world
Reply-To: egreen@east.sun.com
NNTP-Posting-Host: laser.east.sun.com

In article 211353@mavenry.altcit.eskimo.com, maven@mavenry.altcit.eskimo.com (Norman Hamer) writes:
> 
> The question for the day is re: passenger helmets, if you don't know for 
>certain who's gonna ride with you (like say you meet them at a .... church 
>meeting, yeah, that's the ticket)... What are some guidelines? Should I just 
>pick up another shoei in my size to have a backup helmet (XL), or should I 
>maybe get an inexpensive one of a smaller size to accomodate my likely 
>passenger? 

If your primary concern is protecting the passenger in the event of a
crash, have him or her fitted for a helmet that is their size.  If your
primary concern is complying with stupid helmet laws, carry a real big
spare (you can put a big or small head in a big helmet, bu

In [ ]:
from sklearn.datasets import fetch_20newsgroups
# subset = 'train'으로 학습용 데이터만 추출
# remove = ('headers','footers','quotes') 내용만 추출
train_news = fetch_20newsgroups(subset='train',
                                remove=('headers','footers','quotes'),
                                random_state=156)
X_train = train_news.data
y_train = train_news.target

# subset = 'train'으로 학습용 데이터만 추출
# remove = ('headers','footers','quotes') 내용만 추출
test_news = fetch_20newsgroups(subset='test',
                                remove=('headers','footers','quotes'),
                                random_state=156)
X_test = test_news.data
y_test = test_news.target

print('학습데이터크기: {0}, 테스트 데이터 크기: {1}'.format(len(train_news.data),len(test_news.data)))

학습데이터크기: 11314, 테스트 데이터 크기: 7532


## 피처 벡터화 변환과 머신러닝 모델 학습/예측/평가

⚠️ CountVectorizer - test set
- 테스트 데이터에서 CountVectorizer적용시 학습데이터에 사용된 객체를 이용해 transform시켜야 함.
- fit_transform 사용하면 안됨 ( 학습시 사용된 피처 개수와 예측시 사용할 피처 수 달라짐.)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# 피처 벡터화
cnt_vect = CountVectorizer()
cnt_vect.fit(X_train)
X_train_cnt_vect = cnt_vect.transform(X_train)

# 학습데이터로 fit()된 CountVectorizer객체로 테스트 데이터 피처벡터화 변환(.transform())
X_test_cnt_vect = cnt_vect.transform(X_test)

print('학습데이터 텍스트의 CountVectorizer Shape:',X_train_cnt_vect.shape)

학습데이터 텍스트의 CountVectorizer Shape: (11314, 101631)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Logistic Regression - 학습/예측/평가

lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_cnt_vect,y_train)
pred = lr_clf.predict(X_test_cnt_vect)
print('CountVectorized Logistic Regression 예측 정확도 : ',round(accuracy_score(y_test,pred),3))

CountVectorized Logistic Regression 예측 정확도 :  0.617


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 피처 벡터화
tfidf_vect = TfidfVectorizer()
tfidf_vect.fit(X_train)
X_train_tfidf_vect = tfidf_vect.transform(X_train)

# 학습데이터로 fit()된 CountVectorizer객체로 테스트 데이터 피처벡터화 변환(.transform())
X_test_tfidf_vect = tfidf_vect.transform(X_test)

print('학습데이터 텍스트의 TfidfVectorizer Shape:',X_train_tfidf_vect.shape)

lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_tfidf_vect,y_train)
pred = lr_clf.predict(X_test_tfidf_vect)
print('TF-IDF Vectorized Logistic Regression 예측 정확도 : ',round(accuracy_score(y_test,pred),3))

학습데이터 텍스트의 TfidfVectorizer Shape: (11314, 101631)
TF-IDF Vectorized Logistic Regression 예측 정확도 :  0.678


- 예측도 : TF-IDF > Count
- 문서내 텍스트가 많고 많은 문서를 가지는 텍스트 분석에서 TF-IDF가 Count보다 더 좋은 예측결과를 도출함

### 📌텍스트 분석에서 머신러닝 모델의 성능을 향상 시키는 중요한 방법 2가지

- 최적의 ML 알고리즘 선택
- 최상의 피처 전처리 수행

  *   텍스트 정규화
  *   Count/TF-IDF 기반 피처 벡터화



※ TfidfVectorizer 파라미터 적용
- stop_words : None > english
- ngram_range : (1,1)(default) > (1,2)
- max_df = 300

In [ ]:
# stop words 필터링을 추가하고 ngram을 기본(1,1)에서 (1,2)로 변경하여 Feature Vectorization 적용.
tfidf_vect = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_df=300 )
tfidf_vect.fit(X_train)
X_train_tfidf_vect = tfidf_vect.transform(X_train)
X_test_tfidf_vect = tfidf_vect.transform(X_test)

lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_tfidf_vect , y_train)
pred = lr_clf.predict(X_test_tfidf_vect)
print('TF-IDF Vectorized Logistic Regression 의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test ,pred)))

TF-IDF Vectorized Logistic Regression 의 예측 정확도는 0.690


- GridSearchCV : 하이퍼 파리미터 최적화 수행

In [ ]:
from sklearn.model_selection import GridSearchCV

# 최적 C 값 도출 튜닝 수행. CV는 3 Fold셋으로 설정.
params = { 'C':[0.01, 0.1, 1, 5, 10]}
grid_cv_lr = GridSearchCV(lr_clf ,param_grid=params , cv=3 , scoring='accuracy' , verbose=1 )
grid_cv_lr.fit(X_train_tfidf_vect , y_train)
print('Logistic Regression best C parameter :',grid_cv_lr.best_params_ )

# 최적 C 값으로 학습된 grid_cv로 예측 수행하고 정확도 평가.
pred = grid_cv_lr.predict(X_test_tfidf_vect)
print('TF-IDF Vectorized Logistic Regression 의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test ,pred)))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Logistic Regression best C parameter : {'C': 10}
TF-IDF Vectorized Logistic Regression 의 예측 정확도는 0.704


## 사이킷런 파이프라인 사용 및 GridSearchCV와의 결합

📌 머신러닝에서 Pipeline
- 전처리(데이터 가공,변환 등) + 알고리즘 적용이 한꺼번에 물흐르듯 스트림 기반으로 처리
- 데이터 전처리 + Estimator

In [ ]:
from sklearn.pipeline import Pipeline

# TfidfVectorizer 객체를 tfidf_vect 객체명으로, LogisticRegression객체를 lr_clf 객체명으로 생성하는 Pipeline생성
pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_df=300)),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))
])

# 별도의 TfidfVectorizer객체의 fit_transform( )과 LogisticRegression의 fit(), predict( )가 필요 없음.
# pipeline의 fit( ) 과 predict( ) 만으로 한꺼번에 Feature Vectorization과 ML 학습/예측이 가능.
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
print('Pipeline을 통한 Logistic Regression 의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test ,pred)))

Pipeline을 통한 Logistic Regression 의 예측 정확도는 0.704


- 아래 코드를 실행했더니 20분후에 제 노트북이 멈춰서....... 이 코드는 제 노트북 성능으로는 안되는 것 같아 실행 결과를 내지는 못했습니다...

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english')),
    ('lr_clf', LogisticRegression(solver='liblinear'))
])

# Pipeline에 기술된 각각의 객체 변수에 언더바(_)2개를 연달아 붙여 GridSearchCV에 사용될
# 파라미터/하이퍼 파라미터 이름과 값을 설정. .
params = { 'tfidf_vect__ngram_range': [(1,1), (1,2), (1,3)],
           'tfidf_vect__max_df': [100, 300, 700],
           'lr_clf__C': [1, 5, 10]
}

# GridSearchCV의 생성자에 Estimator가 아닌 Pipeline 객체 입력
grid_cv_pipe = GridSearchCV(pipeline, param_grid=params, cv=3 , scoring='accuracy',verbose=1)
grid_cv_pipe.fit(X_train , y_train)
print(grid_cv_pipe.best_params_ , grid_cv_pipe.best_score_)

pred = grid_cv_pipe.predict(X_test)
print('Pipeline을 통한 Logistic Regression 의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test ,pred)))

Fitting 3 folds for each of 27 candidates, totalling 81 fits
